# (노트) Grad-CAM

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [빅데이터분석]

### import

In [10]:
from fastai.vision.all import * 
import torch 

In [11]:
def search_images_ddg(key,max_n=500):
    """Search for 'key' with DuckDuckGo and return a unique urls of 'max_n' images
       (Adopted from https://github.com/deepanprabhu/duckduckgo-images-api)
    """
    url        = 'https://duckduckgo.com/'
    params     = {'q':key}
    res        = requests.post(url,data=params)
    searchObj  = re.search(r'vqd=([\d-]+)\&',res.text)
    if not searchObj: print('Token Parsing Failed !'); return
    requestUrl = url + 'i.js'
    headers    = {'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:71.0) Gecko/20100101 Firefox/71.0'}
    params     = (('l','us-en'),('o','json'),('q',key),('vqd',searchObj.group(1)),('f',',,,'),('p','1'),('v7exp','a'))
    urls       = []
    while True:
        try:
            res  = requests.get(requestUrl,headers=headers,params=params)
            data = json.loads(res.text)
            for obj in data['results']:
                urls.append(obj['image'])
                max_n = max_n - 1
                if max_n < 1: return L(set(urls))     # dedupe
            if 'next' not in data: return L(set(urls))
            requestUrl = url + data['next']
        except:
            pass

In [ ]:
# class Hook(): ## 상태를 기록하는 클래스 
#     def hook_func(self,module, input, output): self.stored = output 

### data 

In [ ]:
keywords = 'pomeranian', 'gray wolf' 
path=Path('GradCAM')

In [ ]:
if not path.exists(): # 현재폴더에 GradCAM라는 폴더가 있는지 체크 
    path.mkdir() # 현재폴더에 GradCAM라는 폴더가 만들어짐 
    for keyword in keywords: # keyword='dog', keyword='wolf' 일때 아래내용을 반복 
        lastpath=path/keyword # ./GradCAM/dog or ./GradCAM/wolf 
        lastpath.mkdir(exist_ok=True) # make ./GradCAM/dog or ./singer/wolf 
        urls=search_images_ddg(keyword) # 'dog','wolf' 검색어로 url들의 리스트를 얻음
        download_images(lastpath,urls=urls) # 그 url에 해당하는 이미지들을  ./GradCAM/dog or ./GradCAM/wolf 에 저장

In [ ]:
verify_images(get_image_files(path)) # 이상한 그림파일 리스트확인

In [ ]:
verify_images(get_image_files(path)).map(Path.unlink) # 이상한 그림파일 삭제

In [ ]:
verify_images(get_image_files(path)) # 다시 리스트확인 

In [ ]:
dls = ImageDataLoaders.from_folder(
    path,
    train='GradCAM',
    valid_pct=0.2, 
    item_tfms=Resize(224))   

In [ ]:
dls.show_batch(max_n=16)

### learn 

In [ ]:
lrnr=cnn_learner(dls,resnet34,metrics=error_rate)
lrnr.fine_tune(5)

In [ ]:
lrnr.show_results(max_n=16)

### Grad-CAM 

In [ ]:
class Hook():
    def __init__(self,m):
        self.hook = m.register_forward_hook(self.hook_func)        
    def hook_func(self,m,i,o): self.stored = o[0].detach().clone() ## 0 is observation index 
    def __enter__(self,*args): return self 
    def __exit__(self,*args): self.hook.remove()    

In [ ]:
class HookBwd():
    def __init__(self,m):
        self.hook = m.register_backward_hook(self.hook_func)
    def hook_func(self,m,gi,go): self.stored = go[0].detach().clone()
    def __enter__(self,*args): return self 
    def __exit__(self,*args): self.hook.remove()

In [ ]:
img=PILImage.create(get_image_files(path)[7])
x,  = first(dls.test_dl([img]))

with HookBwd(lrnr.model[0][-3]) as hookg: # lrnr.model[0] = 1st networks 
    with Hook(lrnr.model[0][-3]) as hook: # lrnr.model[0] = 1st networks 
        output= lrnr.model(x)
        act=hook.stored 
        output[0,0].backward() # 0 is observation index, 1 means wolf //
        grad=hookg.stored 

w=grad[0].mean(dim=[1,2],keepdim=True)
cam_map=(w*act[0]).sum(0)
_,ax= plt.subplots()
x_dec=TensorImage(dls.train.decode((x,))[0][0])
x_dec.show(ctx=ax)
ax.imshow(cam_map.detach().cpu(), alpha=0.7, extent=(0,224,224,0),interpolation='bilinear',cmap='magma');